In [1]:
"""
Rhetorical Role (RR) Extraction Pipeline
=========================================
Stage: RR Extraction (from the architecture diagram)
Model: OpenNyAI InLegalBERT (Rhetorical_Role component)

Input  : JSONL  →  {"id": "2019_144", "text": "...", "label": 0}
Output : JSONL  →  {"id": "2019_144", "Fact": "...", "Issue": "...",
                    "Arg_P": "...", ..., "label": 0}

Each rhetorical-role key maps to the concatenated text of ALL sentences
that were assigned that role. Roles not present in a document are omitted.
"""

import json
import os
import logging
from collections import defaultdict
from pathlib import Path

# ── OpenNyAI imports ──────────────────────────────────────────────────────────
from opennyai import Pipeline
from opennyai.utils import Data

# ── Configuration ─────────────────────────────────────────────────────────────
INPUT_PATH  = "cjpe_train_3col.jsonl"
OUTPUT_PATH = "rr_extracted_output.jsonl"
USE_GPU     = True          # Set False if no GPU available
BATCH_SIZE  = 8             # Lower if you hit OOM errors (e.g. 4 or 2)

# Official rhetorical role labels produced by InLegalBERT
RR_LABEL_ORDER = [
    "Fact",
    "Issue",
    "Arg_P",       # Argument by Petitioner
    "Arg_R",       # Argument by Respondent
    "Analysis",
    "Statute",
    "Ratio",
    "Ruling",
    "None",        # Catch-all for any unlabelled sentence
]

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ─────────────────────────────────────────────────────────────────────────────
# 1. Load dataset
# ─────────────────────────────────────────────────────────────────────────────
def load_jsonl(path: str) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as fh:
        for lineno, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                log.warning(f"Line {lineno}: JSON parse error – {e}")
    log.info(f"Loaded {len(records)} records from '{path}'")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# 2. Build OpenNyAI pipeline (only Rhetorical_Role component needed here)
# ─────────────────────────────────────────────────────────────────────────────
def build_pipeline(use_gpu: bool) -> Pipeline:
    log.info(f"Loading OpenNyAI pipeline  (GPU={use_gpu}) …")
    pipeline = Pipeline(
        components=["Rhetorical_Role"],   # Add 'NER','Summarizer' if needed
        use_gpu=use_gpu,
        verbose=True,
    )
    log.info("Pipeline ready.")
    return pipeline


# ─────────────────────────────────────────────────────────────────────────────
# 3. Process one batch of records
# ─────────────────────────────────────────────────────────────────────────────
def process_batch(pipeline: Pipeline, batch: list[dict]) -> list[dict]:
    """
    Run RR extraction on a batch and return list of output records.

    OpenNyAI Data wraps a list of plain text strings.
    After pipeline execution each doc gets a 'sentences' list where
    every sentence dict has at least:
        { "text": "...", "rhetorical_role": "Fact" | "Issue" | ... }
    """
    texts = [rec["text"] for rec in batch]
    data  = Data(texts)          # wraps list[str] into OpenNyAI data object

    pipeline(data)               # in-place annotation

    output_records = []
    for rec, doc in zip(batch, data):
        # doc is a spaCy Doc (or OpenNyAI wrapper) with sentence spans
        role_buckets: dict[str, list[str]] = defaultdict(list)

        # ── Extract sentences + their rhetorical role ──────────────────────
        # OpenNyAI annotates via doc.user_data or doc.spans; handle both APIs
        sentences = _extract_sentences(doc)

        for sent_text, role in sentences:
            sent_text = sent_text.strip()
            if sent_text:
                role_buckets[role].append(sent_text)

        # ── Build output record ────────────────────────────────────────────
        out = {"id": rec["id"]}

        # Insert role keys in canonical order; skip absent roles
        for role in RR_LABEL_ORDER:
            if role in role_buckets:
                out[role] = " ".join(role_buckets[role])

        # Also store per-sentence detail for downstream use (optional)
        out["rr_sentences"] = [
            {"text": t, "rr_label": r} for t, r in sentences
        ]

        out["label"] = rec["label"]      # preserve original judgment label
        output_records.append(out)

    return output_records


def _extract_sentences(doc) -> list[tuple[str, str]]:
    """
    Robustly extract (sentence_text, rr_label) pairs from an annotated doc.
    Handles both the spaCy Doc interface and raw dict/list formats that
    different OpenNyAI versions may return.
    """
    pairs = []

    # ── Case A: doc is a plain dict (some pipeline versions) ──────────────
    if isinstance(doc, dict):
        for sent in doc.get("sentences", []):
            text = sent.get("text", "")
            role = sent.get("rhetorical_role", "None")
            pairs.append((text, role))
        return pairs

    # ── Case B: doc is a list of sentence dicts ────────────────────────────
    if isinstance(doc, list):
        for sent in doc:
            text = sent.get("text", "")
            role = sent.get("rhetorical_role", "None")
            pairs.append((text, role))
        return pairs

    # ── Case C: spaCy Doc with .sents and custom extension ────────────────
    try:
        for sent in doc.sents:
            role = "None"
            # Custom attribute set by OpenNyAI
            if sent.has_extension("rhetorical_role"):
                role = sent._.rhetorical_role or "None"
            elif sent[0].has_extension("rhetorical_role"):
                role = sent[0]._.rhetorical_role or "None"
            pairs.append((sent.text, role))
        return pairs
    except Exception:
        pass

    # ── Case D: doc has user_data with sentence list ───────────────────────
    try:
        for sent in doc.user_data.get("sentences", []):
            text = sent.get("text", "")
            role = sent.get("rhetorical_role", "None")
            pairs.append((text, role))
        return pairs
    except Exception:
        pass

    # Fallback: return full text as single "None"-labelled sentence
    try:
        pairs.append((doc.text, "None"))
    except Exception:
        pairs.append((str(doc), "None"))
    return pairs


# ─────────────────────────────────────────────────────────────────────────────
# 4. Main driver
# ─────────────────────────────────────────────────────────────────────────────
def main():
    # Ensure output directory exists
    Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

    # Load data
    records = load_jsonl(INPUT_PATH)

    # Build pipeline
    pipeline = build_pipeline(USE_GPU)

    # Process in batches and write output incrementally
    total     = len(records)
    processed = 0
    errors    = 0

    with open(OUTPUT_PATH, "w", encoding="utf-8") as out_fh:
        for batch_start in range(0, total, BATCH_SIZE):
            batch = records[batch_start : batch_start + BATCH_SIZE]
            log.info(
                f"Processing batch {batch_start//BATCH_SIZE + 1} "
                f"| records {batch_start+1}–{min(batch_start+BATCH_SIZE, total)} "
                f"of {total}"
            )

            try:
                output_batch = process_batch(pipeline, batch)
                for out_rec in output_batch:
                    out_fh.write(json.dumps(out_rec, ensure_ascii=False) + "\n")
                processed += len(output_batch)

            except Exception as exc:
                log.error(f"Batch {batch_start//BATCH_SIZE + 1} failed: {exc}")
                # Write partial records with error flag so no data is lost
                for rec in batch:
                    fallback = {
                        "id":    rec["id"],
                        "error": str(exc),
                        "text":  rec["text"],   # keep original text
                        "label": rec["label"],
                    }
                    out_fh.write(json.dumps(fallback, ensure_ascii=False) + "\n")
                errors += len(batch)

    log.info("=" * 60)
    log.info(f"Done.  Processed: {processed}  |  Errors: {errors}")
    log.info(f"Output written to: {OUTPUT_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# 5. Quick sanity check  (run with:  python rr_extraction.py --test)
# ─────────────────────────────────────────────────────────────────────────────
def test_single():
    """
    Smoke-test on a single hard-coded record so you can verify the
    output shape before running on the full dataset.
    """
    dummy = [
        {
            "id": "TEST_001",
            "text": (
                "The appellant was convicted by the Sessions Court under Section 302 IPC. "
                "The main issue before this court is whether the conviction is sustainable. "
                "The counsel for the petitioner argued that the evidence is insufficient. "
                "The State contended that the witnesses are reliable. "
                "This court finds that the lower court's reasoning is sound. "
                "Section 302 of the Indian Penal Code prescribes punishment for murder. "
                "The ratio of this decision rests on the reliability of eyewitness testimony. "
                "The appeal is accordingly dismissed."
            ),
            "label": 0,
        }
    ]
    pipeline = build_pipeline(USE_GPU)
    result   = process_batch(pipeline, dummy)
    print("\n── Sample output ──────────────────────────────────────")
    print(json.dumps(result[0], indent=2, ensure_ascii=False))


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "--test":
        test_single()
    else:
        main()

09:29:16  INFO  Loaded 32305 records from 'cjpe_train_3col.jsonl'
09:29:16  INFO  Loading OpenNyAI pipeline  (GPU=True) …


ℹ Loading Rhetorical Role...
ℹ Rhetorical Roles will use GPU!


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
09:29:23  INFO  Pipeline ready.
09:29:23  INFO  Processing batch 1 | records 1–8 of 32305
/home/RSlab/.conda/envs/myenv1/lib/python3.10/site-packages/torch

ℹ Pre-processing will happen on CPU!
ℹ Preprocessing rhetorical role model input!!!


100%|█████████████████████████████████████████████████████████████████████████████████████| 8/8 [01:41<00:00, 12.63s/it]
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


...

100%|█████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:03<00:00,  2.10it/s]
11:36:19  INFO  Processing batch 199 | records 1585–1592 of 32305


ℹ Pre-processing will happen on CPU!


11:36:20  ERROR  Batch 199 failed: There was an error while loading en_core_web_trf
 To rectify try running:
 pip install -U https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/STOCK_SPACY_MODELS/en_core_web_trf-3.2.0-py3-none-any.whl
11:36:20  INFO  Processing batch 200 | records 1593–1600 of 32305


ℹ Pre-processing will happen on CPU!
ℹ Preprocessing rhetorical role model input!!!


 25%|█████████████████████▎                                                               | 2/8 [00:09<00:28,  4.69s/it]

KeyboardInterrupt

